# Finish the post-midsem run — resume both final models + test evaluation

You already have: (1) the completed 12-run sweep (zip from Nov 16) and (2) the two
150k from-scratch final runs interrupted at **116,500** (control) and **123,500**
(SNR-guided, alpha 4) iterations, with resumable `latest_state.pth` checkpoints
(zip from Nov 17). No test evaluation has been done yet.

This notebook resumes both runs to 150k and then evaluates them once on the held-out
LOL-v2-syn test set. Budget on **one T4: ~5 h** (control 33,500 iters x 0.305 s = 2.8 h;
guided 26,500 x 0.288 s = 2.1 h; plus ~15 min setup/eval). That fits a 6 GPU-hour quota.
Do **not** use 2 GPUs unless your quota is measured in wall-clock time, because Kaggle
usually charges GPU-hours per accelerator.

**Before running:** push the latest repo commit (it contains the `--resume` log-header
fix) to GitHub — this notebook clones the remote. Attach the Nov 17 zip
(`post_midsem_state .zip`) as a Kaggle dataset input. Save versions when each run finishes.


In [ ]:
import os, sys, subprocess, glob, json, zipfile, time, shutil
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Crazycraftie/low-light-enhancement-btp.git', 'btp'], check=True)
os.chdir('/kaggle/working/btp')
print('Repo HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
if not Path('baselines/Retinexformer').exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caiyuanhao1998/Retinexformer.git', 'baselines/Retinexformer'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'lmdb', 'yacs',
                'addict', 'future', 'natsort', 'yapf', 'thop', 'timm', 'lpips', 'scikit-image'], check=True)

import numpy as np
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU first.'
# The interrupted runs used the LOL-v1 yml; its network_g and padding factor are identical
# to the syn yml, so keep the same path to preserve the provenance in config.json.
OPT = 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'
RUNS = Path('runs/post_midsem/final')
assert torch.cuda.device_count() >= 1


## 1. Restore the interrupted checkpoints

The attached zip has the same folder structure as the original notebook, so extracting it
into the repo root restores both runs. The space in the filename is handled by the glob.


In [ ]:
RUNS = Path('runs/post_midsem/final')


def has_final_checkpoints(path):
    with zipfile.ZipFile(path) as zf:
        return 'runs/post_midsem/final/control/latest_state.pth' in zf.namelist()


# Case 1: the zip was attached as a dataset file.
archives = glob.glob('/kaggle/input/**/post_midsem_state*.zip', recursive=True)
archives = [p for p in archives if has_final_checkpoints(p)]
if archives:
    archive = max(archives, key=lambda p: (os.path.getsize(p), os.path.getmtime(p)))
    with zipfile.ZipFile(archive) as zf:
        zf.extractall('/kaggle/working/btp')
    print('Restored zip from', archive)
else:
    # Case 2: Kaggle unpacked the dataset, so look for the checkpoint files directly.
    states = glob.glob('/kaggle/input/**/runs/post_midsem/final/control/latest_state.pth', recursive=True)
    if states:
        source = Path(states[0]).parents[1]          # .../runs/post_midsem/final
        shutil.copytree(source, RUNS, dirs_exist_ok=True)
        print('Restored extracted checkpoints from', source)
    else:
        print('FILES UNDER /kaggle/input (first 40):')
        for p in sorted(glob.glob('/kaggle/input/**/*', recursive=True))[:40]:
            print(' ', p)
        raise FileNotFoundError(
            'No final checkpoints found. Attach the Nov 17 post_midsem_state .zip as a dataset '
            'input (Add Data -> Your Datasets). The above list shows what Kaggle actually mounted.'
        )

for name in ('control', 'snr_attention_a4'):
    d = RUNS / name
    cfg = json.loads((d / 'config.json').read_text())
    assert (d / 'latest_state.pth').is_file(), f'No resume checkpoint in {d}'
    print(f"{name}: iters={cfg['iters']} lr={cfg['lr']} alpha={cfg['alpha']} "
          f"snr_attention={cfg['snr_attention']} pretrained={cfg.get('pretrained')}")
    assert cfg['iters'] == 150000 and cfg.get('pretrained') is None
BEST_ALPHA = 4


## 2. Data

LOL-v2-syn: 900 train / 100 test pairs. Use the same attached dataset or the authors' zip.


In [ ]:
def gdl(fid, out, tries=4):
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown', '-q', fid, '-O', str(out)]).returncode == 0 and out.exists():
            return
    raise RuntimeError('Download failed: ' + str(out))

DATA = Path('data/LOLv2/Synthetic')
if not (DATA / 'Train/Low').is_dir():
    attached = glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low', recursive=True)
    if attached:
        source = Path(attached[0]).parents[2]
        Path('data').mkdir(exist_ok=True)
        if not Path('data/LOLv2').exists():
            Path('data/LOLv2').symlink_to(source, target_is_directory=True)
        print('Using attached LOL-v2 data:', source)
    else:
        gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U', '/kaggle/tmp/lolv2.zip')
        subprocess.run(['unzip', '-q', '-o', '/kaggle/tmp/lolv2.zip', '-d', 'data'], check=True)
for split, expected in (('Train/Low', 900), ('Test/Low', 100), ('Test/Normal', 100)):
    got = len(list((DATA / split).glob('*')))
    assert got == expected, f'{split}: {got} files, expected {expected}'
print('Data OK')


## 3. Resume training — control first, then SNR-guided (one GPU)

Arguments are **identical to the original notebook** except for `--resume`; changing
`--iters`, `--lr`, `--seed`, `--val-*` or `--checkpoint-every` would break the cosine
schedule and change the experiment. A state zip is written after each finished run, so
an interrupted notebook can be attached to the next version.


In [ ]:
common = ['--loss', 'l1', '--iters', '150000', '--lr', '2e-4', '--batch', '8',
          '--crop', '128', '--seed', '42', '--val-fraction', '0.1', '--val-seed', '2026',
          '--val-every', '5000', '--checkpoint-every', '5000', '--log-every', '500',
          '--data', str(DATA / 'Train'), '--opt', OPT, '--repo', 'baselines/Retinexformer',
          '--device', 'cuda']

def zip_state():
    path = '/kaggle/working/resume_state.zip'
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in RUNS.rglob('*'):
            if p.is_file():
                zf.write(p, p.as_posix())
    print('Saved', path)

# Conservative timeouts just above the measured remaining work: 33500 x 0.305 s and 26500 x 0.288 s.
for name, alpha, timeout in (('control', None, 10800), ('snr_attention_a4', BEST_ALPHA, 8400)):
    out = RUNS / name
    if (out / 'metrics.json').exists():
        print('Already complete:', out)
        continue
    assert (out / 'latest_state.pth').is_file(), f'Missing resume state in {out}'
    cmd = ['timeout', str(timeout), sys.executable, 'scripts/finetune.py'] + common + [
        '--out', str(out), '--resume']
    if alpha is not None:
        cmd += ['--snr-attention', '--alpha', str(alpha)]
    print('Resuming', name, 'with timeout', timeout, 's')
    with (out / 'run.log').open('a') as log_file:
        process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
        code = process.wait()
    print(name, 'exit code', code)
    if code not in (0, 124):
        raise RuntimeError((out / 'run.log').read_text()[-6000:])
    if code == 124:
        print(name, 'timed out: attach resume_state.zip to a new version and rerun this cell.')
    zip_state()


## 4. Final test evaluation — once, after both runs are complete

Uses each run's best validation checkpoint. Alpha was already selected on validation by the
completed sweep, never on this test set. Note that the sweep's control had the *highest*
validation mean (23.77 dB vs 23.74 for alpha 4) — if that ordering persists on test, report
it as a negative result rather than presenting the guided model as an improvement.


In [ ]:
import csv, io, math
from PIL import Image

def infer(weights, inp, output, method, alpha=None):
    cmd = [sys.executable, 'scripts/retinexformer_infer.py', '--repo', 'baselines/Retinexformer',
           '--opt', OPT, '--weights', str(weights), '--input', str(inp), '--output', str(output),
           '--method', method, '--device', '0', '--no-speed-log']
    if alpha is not None:
        cmd += ['--snr-attention', '--alpha', str(alpha)]
    subprocess.run(cmd, check=True)

for name, alpha in (('control', None), ('snr_attention_a4', BEST_ALPHA)):
    run = RUNS / name
    if not (run / 'metrics.json').exists():
        print('Incomplete run, skipping test evaluation:', name)
        continue
    weights = run / 'best.pth'
    if not weights.exists():
        weights = run / 'final.pth'
    method = 'PostMidsem-control' if alpha is None else f'PostMidsem-SNR-a{alpha}'
    output = Path('results/post_midsem') / method / 'LOLv2-syn'
    infer(weights, DATA / 'Test/Low', output, method, alpha)
    for script, extra in (
        ('scripts/evaluate.py', ['--gt', str(DATA / 'Test/Normal')]),
        ('scripts/eval_dark_regions.py',
         ['--gt', str(DATA / 'Test/Normal'), '--low', str(DATA / 'Test/Low')]),
    ):
        subprocess.run([sys.executable, script, '--pred', str(output), '--method', method,
                        '--dataset', 'LOLv2-syn'] + extra, check=True)

with zipfile.ZipFile('/kaggle/working/final_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in (Path('results/results.csv'), Path('results/dark_region.csv')):
        if path.exists():
            zf.write(path, path.name)
    for root in (Path('results/per_image'), Path('results/per_image_dark')):
        for p in root.rglob('*'):
            if p.is_file() and 'PostMidsem' in str(p):
                zf.write(p, p.as_posix())
    for name in ('control', 'snr_attention_a4'):
        for filename in ('config.json', 'metrics.json', 'log.csv'):
            p = RUNS / name / filename
            if p.exists():
                zf.write(p, p.as_posix())
print('Saved /kaggle/working/final_results.zip')
print('Test rows:')
print('\n'.join(l for l in Path('results/results.csv').read_text().splitlines() if 'PostMidsem' in l))


## Done — what to bring back

Download **`final_results.zip`** and import `results.csv`, `dark_region.csv`, the
`per_image*` CSVs and the two run configs/logs into the repo. Then update the report's
"Plan for the Remaining Semester" section, the conclusion, tables and slides with the
actual numbers. ExDark detector fine-tuning is dropped for lack of quota; say so explicitly.
